# Greek Medical Report Anonymization

This notebook anonymizes Greek medical reports directly inside Google Colab.

**How to use**

1. Run the setup cell.
2. Run the settings cell.
3. Run the upload-and-process cell.
4. Review the preview inside Colab.
5. Download the generated `.zip` file.


In [ ]:
#@title Step 1 - Setup
print('Setting up the notebook environment...')

%cd /content
!rm -rf greek-medical-anonymization
!git clone https://github.com/VanessaLislevand/greek-medical-anonymization.git
%cd /content/greek-medical-anonymization
%pip install -q -e ".[ml]"

print('Setup completed.')


In [ ]:
#@title Step 2 - Settings and Model Loading

PROCESSING_MODE = "auto" #@param ["auto", "free_text_only", "template_only"]
MASK_TOKEN = "[REDACTED]" #@param {type:"string"}
DEFAULT_MODEL_DIR = "/content/drive/MyDrive/Archimedes_Anonymization/exported_models/xlmr_phi_final_all_data" #@param {type:"string"}

import sys
from pathlib import Path

from IPython.display import Markdown, display
from google.colab import drive

repo_root = Path('/content/greek-medical-anonymization')
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from greek_med_anonymizer.config import AppConfig, ModelConfig, RuleConfig
from greek_med_anonymizer.pipeline import AnonymizationPipeline

print('Mounting Google Drive...')
drive.mount('/content/drive')

model_dir = Path(DEFAULT_MODEL_DIR)
print(f'Checking model folder: {model_dir}')

if not model_dir.exists():
    custom_path = input('Default model path not found. Paste the Google Drive path to the model folder: ').strip()
    model_dir = Path(custom_path)

if not model_dir.exists():
    raise FileNotFoundError('Model folder not found in Google Drive.')

config = AppConfig(
    input_glob='*.docx',
    output_suffix='.anon.txt',
    mask_token=MASK_TOKEN,
    processing_mode=PROCESSING_MODE,
    rules=RuleConfig(phones=True, patient_ids=True),
    model=ModelConfig(
        enabled=True,
        model_dir=str(model_dir),
        labels_to_mask=['PHI'],
        aggregation_strategy='simple',
    ),
)

pipeline = AnonymizationPipeline(config)

display(Markdown('### Pipeline ready'))
print(f'Processing mode: {PROCESSING_MODE}')
print(f'Mask token: {MASK_TOKEN}')
print(f'Model folder: {model_dir}')
print('Model files found:')
for path in sorted(model_dir.iterdir()):
    if path.is_file():
        print(f'- {path.name}')


In [ ]:
#@title Step 3 - Upload, Process, and Download

import sys
from io import BytesIO
import json
from pathlib import Path
import tempfile
import zipfile

from IPython.display import Markdown, display
from google.colab import files

repo_root = Path('/content/greek-medical-anonymization')
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from greek_med_anonymizer.io_utils import read_input_text


def entity_payload(entities):
    return [
        {
            'start': entity.start,
            'end': entity.end,
            'label': entity.label,
            'text': entity.text,
            'source': entity.source,
        }
        for entity in entities
    ]


def build_output_name(filename: str) -> str:
    path = Path(filename)
    stem = path.stem
    if path.parent != Path('.'):
        safe_parent = '__'.join(path.parent.parts)
        return f'{safe_parent}__{stem}.anon.txt'
    return f'{stem}.anon.txt'


def anonymize_payload(filename: str, payload: bytes):
    suffix = Path(filename).suffix or '.txt'
    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as temp_file:
        temp_file.write(payload)
        temp_path = Path(temp_file.name)

    try:
        text = read_input_text(temp_path)
        result = pipeline.anonymize_text(text)
    finally:
        temp_path.unlink(missing_ok=True)

    return {
        'filename': filename,
        'output_name': build_output_name(filename),
        'anonymized_text': result.anonymized_text,
        'metadata': entity_payload(result.entities),
    }


print('Upload one or more .docx/.txt files, or a .zip archive containing reports.')
uploaded = files.upload()

if not uploaded:
    raise RuntimeError('No files were uploaded.')

output_zip_path = Path('/content/anonymized_outputs.zip')
preview_items = []
processed_count = 0

with zipfile.ZipFile(output_zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as output_archive:
    for filename, payload in uploaded.items():
        suffix = Path(filename).suffix.lower()

        if suffix == '.zip':
            with zipfile.ZipFile(BytesIO(payload)) as source_archive:
                for member_name in source_archive.namelist():
                    if member_name.endswith('/'):
                        continue
                    if Path(member_name).suffix.lower() not in {'.docx', '.txt'}:
                        continue

                    summary = anonymize_payload(member_name, source_archive.read(member_name))
                    output_archive.writestr(summary['output_name'], summary['anonymized_text'])
                    output_archive.writestr(
                        summary['output_name'] + '.json',
                        json.dumps(summary['metadata'], ensure_ascii=False, indent=2),
                    )
                    if len(preview_items) < 3:
                        preview_items.append(summary)
                    processed_count += 1
        elif suffix in {'.docx', '.txt'}:
            summary = anonymize_payload(filename, payload)
            output_archive.writestr(summary['output_name'], summary['anonymized_text'])
            output_archive.writestr(
                summary['output_name'] + '.json',
                json.dumps(summary['metadata'], ensure_ascii=False, indent=2),
            )
            if len(preview_items) < 3:
                preview_items.append(summary)
            processed_count += 1
        else:
            print(f'Skipping unsupported file: {filename}')

display(Markdown(f'### Anonymization completed for {processed_count} file(s)'))
print(f'Output archive: {output_zip_path}')

for item in preview_items:
    display(Markdown(f"#### Preview: {item['filename']}"))
    print(item['anonymized_text'][:1500])
    print('-' * 80)
    print(f"Detected entities: {len(item['metadata'])}")
    print()

files.download(str(output_zip_path))
